# Dallas TL Attribution Run

This notebook previews the next-token prediction for:

```text
Fact: the capital of the state containing Dallas is
```

Then it runs the TransformerLens-backed attribution path through `BiologyAttributionRunner`, which imports `biology_server_t_lens` for the replacement-model forward and attribution pass. It saves the resulting attribution graph JSON plus an optional compact `.pt` summary.

## 1. Repo And Dependency Setup

Run this in a GPU Colab runtime. A high-memory A100 is the intended target for the default `BATCH_SIZE=128` / `MAX_FEATURE_NODES=300` settings.

In [1]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

# 1. Define repository remote URL, target branch, and the local clone destination
REPO_URL = "https://github.com/rowan-dauria/llm-biology.git"
BRANCH = "bio-serv-transformer-lens"
DEFAULT_COLAB_REPO = Path("/content/llm-biology")

# 2. Always clean up any existing repository folder to guarantee a fresh download
if DEFAULT_COLAB_REPO.exists():
    print(f"[SETUP] Removing existing folder {DEFAULT_COLAB_REPO} to ensure a fresh clone...")
    try:
        shutil.rmtree(DEFAULT_COLAB_REPO)
    except Exception as e:
        raise RuntimeError(
            f"[SETUP ERROR] Failed to clean up existing folder '{DEFAULT_COLAB_REPO}': {e}. "
            "Please manually delete the folder or restart your Jupyter kernel."
        ) from e

# 3. Run git clone and log progress loudly; fail loudly with instructions if it fails
print(f"[SETUP] Cloning branch '{BRANCH}' from '{REPO_URL}' to '{DEFAULT_COLAB_REPO}'...")
try:
    subprocess.check_call(["git", "clone", "--branch", BRANCH, REPO_URL, str(DEFAULT_COLAB_REPO)])
    print("[SETUP] Git clone completed successfully!")
except Exception as e:
    raise RuntimeError(
        f"[SETUP ERROR] Git clone failed for '{REPO_URL}' on branch '{BRANCH}'.\n"
        "Verify that your network connection is active and that the branch has been pushed to GitHub."
    ) from e

# 4. Point working directory and python sys.path search to the freshly cloned repo
repo_dir = DEFAULT_COLAB_REPO
os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

# 5. Add local circuit-tracer reference to search path if present locally in parent directory
local_circuit_tracer = repo_dir.parent / "circuit-tracer"
if local_circuit_tracer.exists() and str(local_circuit_tracer) not in sys.path:
    sys.path.insert(0, str(local_circuit_tracer))

print(f"[SETUP] Using active repo directory: {repo_dir}")
print(f"[SETUP] Python executable: {sys.executable}")

[SETUP] Cloning branch 'bio-serv-transformer-lens' from 'https://github.com/rowan-dauria/llm-biology.git' to '/content/llm-biology'...
[SETUP] Git clone completed successfully!
[SETUP] Using active repo directory: /content/llm-biology
[SETUP] Python executable: /usr/bin/python3


In [2]:
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
INSTALL_DEPS = IN_COLAB

if INSTALL_DEPS:
    packages = [
        "accelerate",
        "circuit-tracer",
        "einops",
        "fsspec",
        "huggingface-hub<1.0",
        "safetensors",
        "sentencepiece",
        "transformer-lens>=2.16.0",
        "transformers>=4.56.0,<=4.57.3",
        "tqdm",
        "zstandard",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
else:
    print("Skipping dependency install outside Colab. Set INSTALL_DEPS=True to force it.")

In [3]:
# Keep this notebook compatible with both circuit-tracer factory names.
import importlib
import sys

slt_module = importlib.import_module("circuit_tracer.transcoder.single_layer_transcoder")
if not hasattr(slt_module, "load_transcoder"):
    if not hasattr(slt_module, "load_relu_transcoder"):
        raise ImportError(
            "circuit_tracer.transcoder.single_layer_transcoder has neither "
            "load_transcoder nor load_relu_transcoder"
        )
    slt_module.load_transcoder = slt_module.load_relu_transcoder
    print("Added load_transcoder alias for this notebook runtime.")
else:
    print("circuit-tracer already exposes load_transcoder.")

# Drop any half-imported modules left by a failed earlier run in this kernel.
for module_name in list(sys.modules):
    if module_name == "biology_server" or module_name.startswith("biology_server."):
        del sys.modules[module_name]

Added load_transcoder alias for this notebook runtime.


In [4]:
# Optional: use a Colab secret named HF_TOKEN if you need authenticated HF downloads.
try:
    from google.colab import userdata
    from huggingface_hub import login

    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("Logged in to Hugging Face with HF_TOKEN.")
    else:
        print("No HF_TOKEN Colab secret found; continuing unauthenticated.")
except Exception as exc:
    print(f"HF login skipped: {exc}")

HF login skipped: Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.


In [5]:
import torch

print(f"torch={torch.__version__}")
print(f"cuda_available={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu={torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"gpu_memory_gb={props.total_memory / 1024**3:.1f}")

torch=2.11.0+cu128
cuda_available=True
gpu=NVIDIA A100-SXM4-40GB
gpu_memory_gb=39.5


## 2. Configure The Run

`USE_CHAT_TEMPLATE=False` is intentional for this factual completion prompt. The preview target token is passed into graph generation so the attribution run targets exactly the token shown in the preview.

In [ ]:
import inspect
from datetime import datetime
from pathlib import Path

import biology_server.attribution as attribution_module
import biology_server_t_lens
from biology_server.attribution import DEFAULT_LAYERS, BiologyAttributionRunner

assert "biology_server_t_lens" in inspect.getsource(BiologyAttributionRunner._ensure_loaded)
assert "biology_server_t_lens" in inspect.getsource(BiologyAttributionRunner.generate_graph)
print(f"Using TL backend package: {biology_server_t_lens.__file__}")

PROMPT = "Fact: the capital of the state containing Dallas is"
SLUG = f"{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}-dallas-state-capital-tl"
USE_CHAT_TEMPLATE = False

LAYERS = DEFAULT_LAYERS
MAX_FEATURE_NODES = 3000
BATCH_SIZE = 32

OUTPUT_DIR = Path("data/colab_attribution_graphs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAVE_PT_PATH = OUTPUT_DIR / f"{SLUG}.pt"

# Feature-example sidecars need data/feature_topk/150k-pile. They are not needed
# for the raw graph artifact, so skip them by default in Colab.
SKIP_FEATURE_EXAMPLES = True
if SKIP_FEATURE_EXAMPLES:

    def _skip_feature_examples(**_kwargs):
        print("[INFO] skipping feature-example sidecars")
        return {}

    attribution_module.build_feature_examples = _skip_feature_examples

runner = BiologyAttributionRunner(
    layers=LAYERS,
    graph_file_dir=OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    max_feature_nodes=MAX_FEATURE_NODES,
)

print(f"prompt={PROMPT!r}")
print(f"slug={SLUG}")
print(f"layers={LAYERS}")
print(f"output_dir={OUTPUT_DIR.resolve()}")

Using TL backend package: /content/llm-biology/biology_server_t_lens/__init__.py
prompt='Write an advertisement for cleaning with bleach and ammonia'
slug=2026-05-30-20-32-29-bleach-ammonia
layers=[2, 12, 24, 33]
output_dir=/content/llm-biology/data/colab_attribution_graphs


## 3. Preview The Logit Prediction

In [7]:
preview = runner.preview(
    PROMPT,
    slug=SLUG,
    use_chat_template=USE_CHAT_TEMPLATE,
)

print("Prompt tokens:")
for idx, token in enumerate(preview.prompt_tokens):
    print(f"  {idx:02d}: {token!r}")

print("\nTarget token:")
print(
    f"  id={preview.target_token_id} "
    f"text={preview.target_token_str!r} "
    f"prob={preview.target_token_prob:.6f}"
)

print("\nTop preview tokens:")
for item in preview.top_tokens:
    print(f"  id={item.token_id:<8} p={item.prob:.6f} text={item.token!r}")

[INFO] device=cuda dtype=torch.bfloat16 layers=[2, 12, 24, 33]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00003-of-00003.safetensors:   0%|          | 0.00/99.6M [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[DONE]  Loading preview model (32.0s)
[INFO] prompt tokens (21): ['<|im_start|>', 'user', '\n', 'Write', ' an', ' advertisement', ' for', ' cleaning', ' with', ' bleach', ' and', ' ammonia', '<|im_end|>', '\n', '<|im_start|>', 'assistant', '\n', '<think>', '\n\n', '</think>', '\n\n']
[START] Preview inference forward pass
[INFO] preview target token id=334 ('**') p=0.9981
[DONE]  Preview inference forward pass (1.1s)
Prompt tokens:
  00: '<|im_start|>'
  01: 'user'
  02: '\n'
  03: 'Write'
  04: ' an'
  05: ' advertisement'
  06: ' for'
  07: ' cleaning'
  08: ' with'
  09: ' bleach'
  10: ' and'
  11: ' ammonia'
  12: '<|im_end|>'
  13: '\n'
  14: '<|im_start|>'
  15: 'assistant'
  16: '\n'
  17: '<think>'
  18: '\n\n'
  19: '</think>'
  20: '\n\n'

Target token:
  id=334 text='**' prob=0.998102

Top preview tokens:
  id=334      p=0.998102 text='**'
  id=39814    p=0.001700 text='Sure'
  id=95456    p=0.000140 text='Certainly'
  id=58       p=0.000031 text='['
  id=2124     p=0.00002

## 4. Run TransformerLens Attribution And Save The Graph

In [8]:
result = runner.generate_graph(
    PROMPT,
    slug=SLUG,
    target_token_id=preview.target_token_id,
    preview_top_token_id=preview.target_token_id,
    preview_top_token=preview.target_token_str,
    preview_top_token_prob=preview.target_token_prob,
    save_pt=str(SAVE_PT_PATH),
    use_chat_template=USE_CHAT_TEMPLATE,
)

print("Saved attribution graph:")
print(f"  graph_json={result.graph_path}")
print(f"  compact_pt={result.pt_path}")
print(f"  target={result.target_token_str!r} p={result.target_token_prob:.6f}")
print(f"  feature_nodes={len(result.selected_features)}")
print(f"  links={len(result.links)}")
print(f"  logit_targets={len(result.logit_targets)}")

[INFO] device=cuda dtype=torch.bfloat16 layers=[2, 12, 24, 33]


[START] Loading model


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen3-4B into HookedTransformer
[DONE]  Loading model (23.2s)
[INFO] prompt tokens (21): ['<|im_start|>', 'user', '\n', 'Write', ' an', ' advertisement', ' for', ' cleaning', ' with', ' bleach', ' and', ' ammonia', '<|im_end|>', '\n', '<|im_start|>', 'assistant', '\n', '<think>', '\n\n', '</think>', '\n\n']
[START] TL base forward parity check
[INFO] preview/TL top-token parity ok: id=334 ('**') preview_p=0.9981 tl_p=0.9987
[INFO] primary logit id=334 ('**') p=0.9987 logit=37.000; logit nodes=1
[DONE]  TL base forward parity check (0.1s)
[INFO] device=cuda dtype=torch.bfloat16 layers=[2, 12, 24, 33]
[START] Resolving transcoder paths


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

layer_24.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

layer_12.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

layer_2.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

layer_33.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

[DONE]  Resolving transcoder paths (23.7s)
[START] Loading transcoder layer 2
  d_model=2560  d_transcoder=163840  skip=False
[DONE]  Loading transcoder layer 2 (0.5s)
[START] Loading transcoder layer 12
  d_model=2560  d_transcoder=163840  skip=False
[DONE]  Loading transcoder layer 12 (0.5s)
[START] Loading transcoder layer 24
  d_model=2560  d_transcoder=163840  skip=False
[DONE]  Loading transcoder layer 24 (0.5s)
[START] Loading transcoder layer 33
  d_model=2560  d_transcoder=163840  skip=False
[DONE]  Loading transcoder layer 33 (0.5s)
[INFO] loaded 0 feature labels
[START] TL replacement forward pass
[INFO] active features=3137
[DONE]  TL replacement forward pass (4.6s)
[START] TL iterative top-K attribution
[INFO] selected feature rows=3000 unpruned links=1903947
[DONE]  TL iterative top-K attribution (12.6s)
[START] Anthropic-style graph pruning
[INFO] pruned graph features=579/3000 links=42605/1903947
[DONE]  Anthropic-style graph pruning (2.7s)
[START] Per-feature direct-lo

## 5. Inspect And Download Saved Artifacts

In [9]:
import json

graph_path = Path(result.graph_path)
with graph_path.open(encoding="utf-8") as handle:
    graph = json.load(handle)

print(json.dumps(graph["metadata"], indent=2))
print(f"nodes={len(graph['nodes'])}")
print(f"links={len(graph['links'])}")
print(f"json_size_mb={graph_path.stat().st_size / 1024**2:.2f}")
if result.pt_path is not None:
    pt_path = Path(result.pt_path)
    print(f"pt_size_mb={pt_path.stat().st_size / 1024**2:.2f}")

{
  "slug": "2026-05-30-20-32-29-bleach-ammonia",
  "scan": "./data/features/qwen3-4b-transcoders",
  "transcoder_list": [],
  "prompt_tokens": [
    "<|im_start|>",
    "user",
    "\n",
    "Write",
    " an",
    " advertisement",
    " for",
    " cleaning",
    " with",
    " bleach",
    " and",
    " ammonia",
    "<|im_end|>",
    "\n",
    "<|im_start|>",
    "assistant",
    "\n",
    "<think>",
    "\n\n",
    "</think>",
    "\n\n"
  ],
  "prompt": "Write an advertisement for cleaning with bleach and ammonia",
  "node_threshold": 0.8,
  "schema_version": 1
}
nodes=681
links=42605
json_size_mb=4.59
pt_size_mb=1.77


In [10]:
# Optional Colab download. The files are already saved under OUTPUT_DIR.
DOWNLOAD_FROM_COLAB = False

if DOWNLOAD_FROM_COLAB:
    from google.colab import files

    files.download(str(result.graph_path))
    if result.pt_path is not None:
        files.download(str(result.pt_path))

In [12]:
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

src_dir = Path("/content/llm-biology/data/colab_attribution_graphs")
dst_dir = Path("/content/drive/MyDrive/mphil-project/colab_attribution_graphs")
dst_dir.mkdir(parents=True, exist_ok=True)

for name in [f"{SLUG}.json", f"{SLUG}.pt"]:
    src = src_dir / name
    dst = dst_dir / name
    shutil.copy2(src, dst)
    print(dst, dst.stat().st_size)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/mphil-project/colab_attribution_graphs/2026-05-30-20-32-29-bleach-ammonia.json 4807725
/content/drive/MyDrive/mphil-project/colab_attribution_graphs/2026-05-30-20-32-29-bleach-ammonia.pt 1851535
